<a href="https://colab.research.google.com/github/fauzanrahmatullah/information-retrieval/blob/main/information-retrieval-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git config --global user.email "fauzan1464@gmail.com"
!git config --global user.name "fauzanrahmatullah"

In [ ]:
import pandas as pd
import numpy as np
import math
import re
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/IR/dataset_artikel_preprocessed.csv', sep=None, engine='python', encoding='latin-1')

documents = {}
doc_lengths = {}
total_doc_length = 0

for doc_id, text in enumerate(df['tokens_hasil_praproses']):
    if pd.notna(text):
      #cleaning stirng
        clean_text = re.sub(r'[^a-z0-9\s]', '', str(text).lower())
        tokens = clean_text.split()
        documents[doc_id] = tokens
        doc_lengths[doc_id] = len(tokens)
        total_doc_length += len(tokens)
    else:
        documents[doc_id] = []
        doc_lengths[doc_id] = 0

N = len(documents) #total jumlah dokumen di korpus
avgdl = total_doc_length / N if N > 0 else 0 #rata-rata panjang dokumen

#indexing
#term -> (df, [docID_1, docID_2, dst...])
standard_index = {}
# term_freqs: docID -> (term: frekuensi term di dokumen tsb)
term_freqs = defaultdict(lambda: defaultdict(int))
temp_index = defaultdict(set)

for doc_id, tokens in documents.items():
    for token in tokens:
        term_freqs[doc_id][token] += 1
        temp_index[token].add(doc_id)

for term, doc_set in sorted(temp_index.items()):
    sorted_postings = sorted(list(doc_set))
    df_val = len(sorted_postings)
    standard_index[term] = (df_val, sorted_postings)

#TF-IDF & COSINE SIMILARITY
def hitung_idf(df_val, N):
    """Menghitung IDF menggunakan logaritma basis 10"""
    if df_val == 0:
        return 0
    return math.log10(N / df_val)

def hitung_tf_weight(tf):
    """Menghitung bobot logaritmik TF: 1 + log10(tf)"""
    if tf > 0:
        return 1 + math.log10(tf)
    return 0

# Hitung dan simpan vektor panjang tiap dokumen (untuk normalisasi Cosine)
doc_vector_lengths = {}
for doc_id, terms_dict in term_freqs.items():
    sum_squares = 0
    for term, tf in terms_dict.items():
        if term in standard_index:
            df_val = standard_index[term][0]
            idf = hitung_idf(df_val, N)
            tf_weight = hitung_tf_weight(tf)
            weight = tf_weight * idf
            sum_squares += weight ** 2
    doc_vector_lengths[doc_id] = math.sqrt(sum_squares)

def rank_tfidf(query_str):
    """Mencari dan merangking dokumen menggunakan Cosine TF-IDF"""
    #cleaning query
    query_tokens = re.sub(r'[^a-z0-9\s]', '', str(query_str).lower()).split()

    #bobot term query
    query_weights = {}
    sum_query_squares = 0
    for term in set(query_tokens):
        if term in standard_index:
            df_val = standard_index[term][0]
            idf = hitung_idf(df_val, N)
            #tf_query = 1, bobot_tf_query = 1 + log10(1) = 1
            weight = 1 * idf
            query_weights[term] = weight
            sum_query_squares += weight ** 2

    query_len = math.sqrt(sum_query_squares)
    if query_len == 0:
        return [] # Semua term query tidak ada di indeks

    #hitung Cos Similarity
    scores = defaultdict(float)
    for term, q_weight in query_weights.items():
        if term in standard_index:
            postings = standard_index[term][1]
            for doc_id in postings:
                tf_doc = term_freqs[doc_id][term]
                idf = hitung_idf(standard_index[term][0], N)
                doc_weight = hitung_tf_weight(tf_doc) * idf
                # Dot product
                scores[doc_id] += (q_weight * doc_weight)

    #normalisasi (bagi dengan panjang vektor query dan dokumen)
    results = []
    for doc_id, dot_product in scores.items():
        d_len = doc_vector_lengths.get(doc_id, 1)
        if d_len > 0:
            cosine_sim = dot_product / (query_len * d_len)
            results.append((doc_id, cosine_sim))

    #urutkan skor tertinggi ke terendah
    results.sort(key=lambda x: x[1], reverse=True)
    return results

#BM25
def rank_bm25(query_str, k1=1.2, b=0.75):
    """Mencari dan merangking dokumen menggunakan algoritma BM25"""
    query_tokens = re.sub(r'[^a-z0-9\s]', '', str(query_str).lower()).split()
    scores = defaultdict(float)

    for term in set(query_tokens):
        if term in standard_index:
            df_val = standard_index[term][0]
            #IDF probabilistik pada BM25 log(N/df)
            idf = math.log10(N / df_val)
            postings = standard_index[term][1]

            for doc_id in postings:
                tf_doc = term_freqs[doc_id][term]
                dl = doc_lengths.get(doc_id, 0)

                #kejenuhan TF & normalisasi panjang dokumen
                numerator = tf_doc * (k1 + 1)
                denominator = tf_doc + k1 * (1 - b + b * (dl / avgdl))
                term_score = idf * (numerator / denominator)

                scores[doc_id] += term_score

    results = list(scores.items())
    results.sort(key=lambda x: x[1], reverse=True)
    return results

#testing
daftar_query = [
    "harga mobil listrik", "bursa saham eropa", "kredit bank mandiri",
    "infrastruktur jalan tol", "teknologi kecerdasan buatan",
    "menteri keuangan sri mulyani", "vaksinasi covid booster",
    "pertumbuhan ekonomi asia", "harga minyak mentah", "pembangunan ibu kota negara"
]

top_k = 3

print(f"HASIL UJI COBA {len(daftar_query)} QUERY\ndiambil Top {top_k} dengan (DocID, skor kemiripan)")

for i, q in enumerate(daftar_query, 1):
    print(f"\nQuery {i}: '{q}'")

    #TF-IDF
    res_tfidf = [(doc_id, round(score, 3)) for doc_id, score in rank_tfidf(q)[:top_k]]
    print(f"  [TF-IDF]                       : {res_tfidf}")

    #BM25 Baseline k1=1.2, b=0.75)
    res_bm25_base = [(doc_id, round(score, 3)) for doc_id, score in rank_bm25(q, 1.2, 0.75)[:top_k]]
    print(f"  [BM25 Baseline k1=1.2, b=0.75] : {res_bm25_base}")

    #Variasi b=0.0
    res_bm25_b0 = [(doc_id, round(score, 3)) for doc_id, score in rank_bm25(q, 1.2, 0.0)[:top_k]]
    print(f"  [BM25 b Nol k1=1.2, b=0.0]     : {res_bm25_b0}")

    #Variasi b=1.0
    res_bm25_b1 = [(doc_id, round(score, 3)) for doc_id, score in rank_bm25(q, 1.2, 1.0)[:top_k]]
    print(f"  [BM25 b Penuh k1=1.2, b=1.0]   : {res_bm25_b1}")

    #Variasi k1 Rendah (kejenuhan cepat)
    res_bm25_k_low = [(doc_id, round(score, 3)) for doc_id, score in rank_bm25(q, 0.5, 0.75)[:top_k]]
    print(f"  [BM25 k1 Rendah k1=0.5, b=0.75]: {res_bm25_k_low}")

    #Variasi k1 Tinggi (kejenuhan lambat)
    res_bm25_k_high = [(doc_id, round(score, 3)) for doc_id, score in rank_bm25(q, 3.0, 0.75)[:top_k]]
    print(f"  [BM25 k1 Tinggi k1=3.0, b=0.75]: {res_bm25_k_high}")

Mounted at /content/drive
HASIL UJI COBA 10 QUERY
diambil Top 3 dengan (DocID, skor kemiripan)

Query 1: 'harga mobil listrik'
  [TF-IDF]                       : [(44, 0.146), (14, 0.074), (7, 0.069)]
  [BM25 Baseline k1=1.2, b=0.75] : [(44, 2.177), (49, 2.104), (13, 1.755)]
  [BM25 b Nol k1=1.2, b=0.0]     : [(44, 4.805), (14, 4.588), (22, 4.388)]
  [BM25 b Penuh k1=1.2, b=1.0]   : [(49, 1.907), (44, 1.841), (13, 1.548)]
  [BM25 k1 Rendah k1=0.5, b=0.75]: [(44, 2.394), (49, 1.818), (7, 1.659)]
  [BM25 k1 Tinggi k1=3.0, b=0.75]: [(49, 2.482), (44, 2.002), (13, 1.877)]

Query 2: 'bursa saham eropa'
  [TF-IDF]                       : [(34, 0.113), (46, 0.109), (36, 0.042)]
  [BM25 Baseline k1=1.2, b=0.75] : [(46, 1.738), (34, 1.221), (2, 0.45)]
  [BM25 b Nol k1=1.2, b=0.0]     : [(46, 6.054), (34, 5.077), (36, 2.538)]
  [BM25 b Penuh k1=1.2, b=1.0]   : [(46, 1.41), (34, 0.975), (2, 0.355)]
  [BM25 k1 Rendah k1=0.5, b=0.75]: [(46, 2.241), (34, 1.733), (2, 0.652)]
  [BM25 k1 Tinggi k1=3.0,